In [0]:
# ============================================================
# CELL 0 - DEPENDENCY INSTALLATION
# ============================================================
#
# WHAT THIS CELL DOES:
# Installs openpyxl into the serverless notebook environment.
# Serverless compute does not ship with openpyxl by default.
# Without it, pandas cannot read .xlsx files.
#
# WHY %pip AND NOT A CLUSTER LIBRARY:
# Serverless compute has no cluster-level library
# configuration. Package installation is per-notebook via
# %pip. The install persists for the lifetime of the
# notebook session only.
#
# WHY THIS BELONGS AT THE TOP:
# Every downstream cell that reads Excel depends on this
# install. Running it once at the top guarantees the
# dependency is available before Cell 4 tries to read.
# ============================================================

%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ============================================================
# CELL 0b - RESTART PYTHON
# ============================================================
# Required after %pip install so the Python interpreter picks
# up the newly installed package. On serverless the kernel
# restarts quickly.
# ============================================================

dbutils.library.restartPython()

In [0]:
# ============================================================
# NOTEBOOK: nb_02_Bronze
# PURPOSE:  Reads each source file from the raw landing volume
#           using the ingestion configuration, validates the
#           observed schema against the schema registry, and
#           writes one Bronze Delta table per logical source.
#
#           Bronze preserves source values exactly. It adds
#           lineage columns but performs no business
#           transformation. Standardisation happens in Silver.
#
# CATALOG:  ktu_assessment_dev
# COMPUTE:  Serverless
# INPUT:    audit.ingestion_config, audit.schema_registry,
#           volume raw_landing
# OUTPUT:   bronze.capturing_tool, bronze.chw_attendance,
#           bronze.online_export, bronze.lookup_courses,
#           bronze.lookup_facility,
#           audit.pipeline_run_log, audit.data_quality_results
# ============================================================

# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------
import uuid
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------
CATALOG       = "ktu_assessment_dev"
AUDIT_SCHEMA  = "audit"
BRONZE_SCHEMA = "bronze"

VOLUME_PATH  = "/Volumes/ktu_assessment_dev/bronze/raw_landing"
SOURCE_ROOT  = f"{VOLUME_PATH}/Training Data"

DEBUG = 1

# ------------------------------------------------------------
# RUN HEADER
# ------------------------------------------------------------
run_id     = str(uuid.uuid4())
notebook   = "nb_02_Bronze"
start_time = datetime.now()

if DEBUG:
    print("=" * 50)
    print("NB_02_BRONZE STARTED")
    print("=" * 50)
    print(f"Run ID     : {run_id}")
    print(f"Start Time : {start_time}")

NB_02_BRONZE STARTED
Run ID     : a74a64a6-7c9f-45a8-ad11-441503991a26
Start Time : 2026-09-12 13:27:59.336513


In [0]:
# ============================================================
# AUDIT START
# ============================================================
# Writes a RUNNING row before processing begins so the audit
# trail is complete even if the notebook fails partway.
# ============================================================

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    VALUES (
        '{run_id}',
        '{notebook}',
        'bronze',
        'bronze layer',
        '{start_time.strftime("%Y-%m-%d %H:%M:%S")}',
        NULL,
        'RUNNING',
        0,
        0,
        0,
        'Bronze layer in progress',
        0
    )
""")

if DEBUG:
    print(f"Audit RUNNING row written for run_id {run_id}")

Audit RUNNING row written for run_id a74a64a6-7c9f-45a8-ad11-441503991a26


In [0]:
# ============================================================
# SOURCE READERS
# ============================================================
#
# WHAT THIS CELL DOES:
# Defines one reader function per source format. Each reader
# takes the source parameters from ingestion_config and
# returns a DataFrame with source values exactly as read.
#
# WHY SOURCE-SPECIFIC READERS AND NOT ONE GENERIC READER:
# The three source structures are genuinely different:
#   - Capturing tool : xlsx, three sheets, header=0, 37 cols
#   - CHW attendance : xlsx, one sheet, header=1, 49 cols
#   - Online export  : csv, one file, comma-delimited, 27 cols
#   - Lookups        : xlsx, one sheet each, header=0
#
# A single generic reader would need so many conditional
# branches that it would be harder to read than three
# explicit readers. The principle: dynamic where patterns
# are genuinely common, explicit where source differences
# are real.
#
# WHAT IS COMMON AND WHAT IS NOT:
# Common:     source path resolution, lineage column naming,
#             schema validation, Bronze write pattern.
# Not common: file format handling, header row, multi-sheet
#             union, delimiter/encoding.
#
# The common parts live in this cell as shared helpers. The
# format-specific parts are three small explicit functions.
# ============================================================

config_df = spark.sql(f"""
    SELECT source_name, source_file, source_subfolder,
           source_format, sheet_name, header_row,
           target_table, processing_order, notes
    FROM {CATALOG}.{AUDIT_SCHEMA}.ingestion_config
    WHERE active_flag = true
    ORDER BY processing_order
""")

config_rows = config_df.collect()


def resolve_source_path(subfolder, file_name):
    """
    Return the full volume path for a source file.
    """
    if subfolder:
        return f"{SOURCE_ROOT}/{subfolder}/{file_name}"
    return f"{SOURCE_ROOT}/{file_name}"


def read_excel_sheet(file_path, sheet_name, header_row):
    """
    Read a single sheet from an xlsx file with the given
    header row. Returns a pandas DataFrame.

    Why pandas: openpyxl (pandas backend) handles merged
    cells and multi-row headers correctly. Spark's native
    Excel reader does not support these reliably.
    """
    import pandas as pd
    return pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=header_row,
        engine="openpyxl",
    )


def read_csv_file(file_path):
    """
    Read a delimited text file with auto-detected encoding
    and delimiter. Returns a pandas DataFrame.
    """
    import pandas as pd

    encoding_candidates = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]
    delimiter_candidates = [",", ";", "\t", "|"]

    for enc in encoding_candidates:
        for delim in delimiter_candidates:
            try:
                candidate = pd.read_csv(
                    file_path, encoding=enc, sep=delim, nrows=100, low_memory=False
                )
                if len(candidate.columns) > 1:
                    return pd.read_csv(
                        file_path, encoding=enc, sep=delim, low_memory=False
                    )
            except Exception:
                continue

    raise RuntimeError(
        f"Could not determine encoding/delimiter for {file_path}"
    )


def pandas_to_spark(pdf):
    """
    Convert a pandas DataFrame to a Spark DataFrame with all
    columns as strings. Bronze stores source values verbatim;
    type casting is a Silver concern.
    """
    for col in pdf.columns:
        pdf[col] = pdf[col].astype("string")
    return spark.createDataFrame(pdf)


if DEBUG:
    print("=" * 50)
    print("SOURCE READERS READY")
    print("=" * 50)
    print(f"Config rows loaded : {len(config_rows)}")

SOURCE READERS READY
Config rows loaded : 9


In [0]:
# ============================================================
# READ: CAPTURING TOOL
# ============================================================
#
# WHAT THIS CELL DOES:
# Reads the three Capturer sheets from the capturing tool
# workbook and unions them into one DataFrame. Adds lineage
# columns identifying which sheet each row came from.
#
# WHY UNION AND NOT THREE TABLES:
# The three sheets share an identical 37-column schema.
# Treating them as three separate sources would triple the
# number of Bronze tables and downstream join complexity for
# no analytical benefit. One Bronze table with a
# _source_sheet column preserves the distinction without
# fragmenting the pipeline.
#
# WHY APPEND AND NOT SPARK UNION DIRECTLY:
# pandas concatenation is used because the sheets are small
# individually (2,731 / 6,701 / 2,175 rows). One pandas
# concat then one Spark conversion is faster than three
# Spark unions of single-row DataFrames.
# ============================================================

import pandas as pd

capture_config = [r for r in config_rows if r["source_name"] == "capturing_tool"]

capture_frames = []
for cfg in capture_config:
    file_path = resolve_source_path(cfg["source_subfolder"], cfg["source_file"])
    pdf = read_excel_sheet(file_path, cfg["sheet_name"], cfg["header_row"])
    pdf["_source_sheet"] = cfg["sheet_name"]
    capture_frames.append(pdf)

    if DEBUG:
        print(f"  Read {cfg['sheet_name']}: {len(pdf)} rows, {len(pdf.columns)} columns")

capture_pdf = pd.concat(capture_frames, ignore_index=True)

if DEBUG:
    print(f"  Unioned total: {len(capture_pdf)} rows")

capturing_tool_df = pandas_to_spark(capture_pdf)

if DEBUG:
    print(f"  Spark DataFrame: {capturing_tool_df.count()} rows")

  Read Capturer1: 2731 rows, 38 columns
  Read Capturer2: 6701 rows, 38 columns
  Read Capturer3: 2175 rows, 38 columns
  Unioned total: 11607 rows


/home/spark-d1e8cd91-59f0-4ec7-a900-ec/.ipykernel/75/command-8721772635636984-3081816948:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  capture_pdf = pd.concat(capture_frames, ignore_index=True)


  Spark DataFrame: 11607 rows


In [0]:
# ============================================================
# READ: CHW ATTENDANCE
# ============================================================
#
# WHAT THIS CELL DOES:
# Reads the Training Attendance sheet from each of the three
# CHW workbooks and unions them into one DataFrame. Adds
# lineage columns identifying which file each row came from.
#
# WHY HEADER ROW 1 AND NOT 0:
# Row 0 is the printed form title ('Date Submitted: ...').
# Row 1 contains the real column names. This was discovered
# during Jupyter profiling and recorded in ingestion_config
# as header_row = 1.
#
# WHY THE THREE FILES UNION CLEANLY:
# All three workbooks share an identical 49-column schema
# on the Training Attendance sheet. Confirmed during
# profiling.
# ============================================================

chw_config = [
    r for r in config_rows
    if r["source_name"] in ("chw_west_coast", "chw_kess", "chw_witzenberg")
]

chw_frames = []
for cfg in chw_config:
    file_path = resolve_source_path(cfg["source_subfolder"], cfg["source_file"])
    pdf = read_excel_sheet(file_path, cfg["sheet_name"], cfg["header_row"])
    pdf["_source_sheet"] = cfg["source_name"]
    chw_frames.append(pdf)

    if DEBUG:
        print(f"  Read {cfg['source_name']}: {len(pdf)} rows, {len(pdf.columns)} columns")

chw_pdf = pd.concat(chw_frames, ignore_index=True)

if DEBUG:
    print(f"  Unioned total: {len(chw_pdf)} rows")

chw_attendance_df = pandas_to_spark(chw_pdf)

if DEBUG:
    print(f"  Spark DataFrame: {chw_attendance_df.count()} rows")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-d1e8cd91-59f0-4ec7-a900-ec95770ced93/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


  Read chw_west_coast: 22 rows, 50 columns


/local_disk0/.ephemeral_nfs/envs/pythonEnv-d1e8cd91-59f0-4ec7-a900-ec95770ced93/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


  Read chw_kess: 233 rows, 50 columns


/local_disk0/.ephemeral_nfs/envs/pythonEnv-d1e8cd91-59f0-4ec7-a900-ec95770ced93/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/home/spark-d1e8cd91-59f0-4ec7-a900-ec/.ipykernel/75/command-8721772635636988-1929930408:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  chw_pdf = pd.concat(chw_frames, ignore_index=True)
/home/spark-d1e8cd91-59f0-4ec7-a900-ec/.ipykernel/75/command-8721772635636988-1929930408:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior,

  Read chw_witzenberg: 104 rows, 50 columns
  Unioned total: 359 rows
  Spark DataFrame: 359 rows


In [0]:
# ============================================================
# READ: ONLINE EXPORT
# ============================================================
#
# WHAT THIS CELL DOES:
# Reads the online school CSV export and adds lineage
# columns.
#
# WHY ENCODING AND DELIMITER DETECTION:
# Confirmed during profiling as UTF-8, comma-delimited. The
# detection logic is defensive - if the source changes its
# export format in future, the pipeline will detect it
# rather than silently mis-parse.
# ============================================================

online_cfg = next(r for r in config_rows if r["source_name"] == "online_export")
online_path = resolve_source_path(online_cfg["source_subfolder"], online_cfg["source_file"])

online_pdf = read_csv_file(online_path)
online_pdf["_source_sheet"] = "online_export"

if DEBUG:
    print(f"  Read online_export: {len(online_pdf)} rows, {len(online_pdf.columns)} columns")

online_export_df = pandas_to_spark(online_pdf)

if DEBUG:
    print(f"  Spark DataFrame: {online_export_df.count()} rows")

  Read online_export: 25824 rows, 28 columns
  Spark DataFrame: 25824 rows


In [0]:
# ============================================================
# READ: LOOKUP TABLES
# ============================================================
#
# WHAT THIS CELL DOES:
# Reads the standalone Course and Facility lookup sheets
# from 'Course and Facility Look Ups.xlsx'.
#
# WHY THESE ARE BRONZE AND NOT SILVER:
# The lookups are source data. They are ingested and
# preserved exactly. Silver will join them to the fact
# sources and produce conformed dimensions. Bronze keeps
# them separate.
#
# NOTE ON THE EMBEDDED CHW LOOKUPS:
# The three CHW workbooks also contain embedded lookups
# (lu_District, lu_Employer, lu_Facility, lu_General,
# lu_Profession). Those are NOT ingested as separate Bronze
# tables because they are identical across all three files
# and redundant with the standalone lookup file. Their
# content is already captured in the audit mapping tables
# built during setup. Documented as a design decision.
# ============================================================

lookup_courses_cfg = next(r for r in config_rows if r["source_name"] == "lookup_courses")
lookup_facility_cfg = next(r for r in config_rows if r["source_name"] == "lookup_facility")

courses_path = resolve_source_path(
    lookup_courses_cfg["source_subfolder"], lookup_courses_cfg["source_file"]
)
facility_path = resolve_source_path(
    lookup_facility_cfg["source_subfolder"], lookup_facility_cfg["source_file"]
)

courses_pdf = read_excel_sheet(courses_path, lookup_courses_cfg["sheet_name"], 0)
courses_pdf["_source_sheet"] = lookup_courses_cfg["sheet_name"]

facility_pdf = read_excel_sheet(facility_path, lookup_facility_cfg["sheet_name"], 0)
facility_pdf["_source_sheet"] = lookup_facility_cfg["sheet_name"]

if DEBUG:
    print(f"  Read LU_Courses:  {len(courses_pdf)} rows, {len(courses_pdf.columns)} columns")
    print(f"  Read LU_Facility: {len(facility_pdf)} rows, {len(facility_pdf.columns)} columns")

lookup_courses_df  = pandas_to_spark(courses_pdf)
lookup_facility_df = pandas_to_spark(facility_pdf)

  Read LU_Courses:  340 rows, 8 columns
  Read LU_Facility: 809 rows, 20 columns


In [0]:
# ============================================================
# SCHEMA VALIDATION
# ============================================================
#
# WHAT THIS CELL DOES:
# Compares the observed columns in each source DataFrame to
# the expected columns recorded in audit.schema_registry.
#
# CLASSIFICATION:
#   Breaking  - a required column is missing. Fail loudly.
#   Safe      - extra columns appeared that are not in the
#               registry. Log them but continue.
#   Ambiguous - a registry column is missing but not marked
#               required. Log as warning and continue.
#
# WHY THE REGISTRY IS CURRENTLY EMPTY:
# nb_00_Setup created schema_registry with no rows. Populating
# it is a separate step. Until it is populated, this cell
# logs observed columns and treats everything as safe. The
# registry can be populated during documentation review
# without re-running the pipeline.
#
# THE VALIDATION PRINCIPLE STILL APPLIES:
# Even with an empty registry, this cell demonstrates the
# structure that would catch a breaking schema change. When
# the registry is filled in, the same code enforces it.
# ============================================================

from pyspark.sql.types import StructType, StructField, StringType


def validate_schema(spark_df, source_label, expected_columns):
    """
    Compare observed columns to expected. Returns a dict with
    missing_required, extra_observed, and status.
    """
    observed = set(spark_df.columns)
    expected = set(expected_columns) if expected_columns else set()

    missing_required = expected - observed
    extra_observed = observed - expected

    return {
        "source":           source_label,
        "observed_count":   len(observed),
        "expected_count":   len(expected),
        "missing_required": sorted(missing_required),
        "extra_observed":   sorted(extra_observed),
        "status":           "BREAKING" if missing_required else "PASS",
    }


expected_registry = {}
for row in spark.sql(f"""
    SELECT source_name, column_name, is_required
    FROM {CATALOG}.{AUDIT_SCHEMA}.schema_registry
    WHERE is_active = true
""").collect():
    expected_registry.setdefault(row["source_name"], []).append(row["column_name"])


sources_to_validate = [
    ("capturing_tool",   capturing_tool_df),
    ("chw_attendance",   chw_attendance_df),
    ("online_export",    online_export_df),
    ("lookup_courses",   lookup_courses_df),
    ("lookup_facility",  lookup_facility_df),
]

validation_results = []
for label, sdf in sources_to_validate:
    expected_cols = expected_registry.get(label, [])
    result = validate_schema(sdf, label, expected_cols)
    validation_results.append(result)

    # ------------------------------------------------------
    # WRITE A DATA QUALITY ROW PER SOURCE
    # First delete any existing row for this run + source,
    # so re-running the cell does not create duplicates.
    # ------------------------------------------------------
    spark.sql(f"""
        DELETE FROM {CATALOG}.{AUDIT_SCHEMA}.data_quality_results
        WHERE run_id = '{run_id}'
          AND check_name = 'schema_validation_{label}'
    """)

    spark.sql(f"""
        INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.data_quality_results
        VALUES (
            '{run_id}',
            'schema_validation_{label}',
            'schema',
            'bronze.{label}',
            'All required columns present',
            '{result["observed_count"]} observed, {len(result["missing_required"])} missing required',
            '{result["status"]}',
            {repr(", ".join(result["extra_observed"][:5])) if result["extra_observed"] else "NULL"},
            current_timestamp()
        )
    """)

if DEBUG:
    print("=" * 60)
    print("SCHEMA VALIDATION")
    print("=" * 60)
    for r in validation_results:
        status = r["status"]
        marker = "OK   " if status == "PASS" else "FAIL "
        print(f"  {marker} {r['source']:<18} {r['observed_count']} cols  status={status}")
        if r["missing_required"]:
            print(f"         MISSING REQUIRED: {r['missing_required']}")
    print("=" * 60)

# ------------------------------------------------------------
# FAIL LOUDLY ON BREAKING SCHEMA CHANGES
# ------------------------------------------------------------
breaking = [r for r in validation_results if r["status"] == "BREAKING"]
if breaking:
    raise RuntimeError(
        f"Breaking schema changes detected in {len(breaking)} source(s). "
        f"See data_quality_results for details."
    )

SCHEMA VALIDATION
  OK    capturing_tool     38 cols  status=PASS
  OK    chw_attendance     50 cols  status=PASS
  OK    online_export      28 cols  status=PASS
  OK    lookup_courses     8 cols  status=PASS
  OK    lookup_facility    20 cols  status=PASS


In [0]:
# ============================================================
# WRITE BRONZE TABLES
# ============================================================
#
# WHAT THIS CELL DOES:
# Writes each source DataFrame to its Bronze Delta table,
# adding standard lineage columns.
#
# WHY COLUMN MAPPING IS ENABLED:
# Source columns contain spaces, parentheses, commas and
# slashes (e.g. 'Date first captured (YYYY/MM/DD)'). Parquet
# does not permit these characters in physical column names,
# and Delta raises DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES
# without column mapping.
#
# Column mapping mode 'name' allows these characters in
# logical column names while storing them under stable
# physical names in the Parquet files. This preserves the
# source column names verbatim in Bronze, which is the
# correct behaviour for a raw-preserving layer.
#
# WHY OVERWRITE MODE:
# Bronze is a mechanical projection of raw. Rerunning with
# the same raw produces the same Bronze. Overwrite guarantees
# the table reflects current raw regardless of how many
# times the notebook runs.
# ============================================================

bronze_writes = [
    ("capturing_tool",   capturing_tool_df,  "Capturing Tool_V1c V2-27 January 2026_SCRUBBED.xlsx"),
    ("chw_attendance",   chw_attendance_df,  "(three CHW files unioned)"),
    ("online_export",    online_export_df,   "Online Data Export 17 Dec_SCRUBBED.csv"),
    ("lookup_courses",   lookup_courses_df,  "Course and Facility Look Ups.xlsx"),
    ("lookup_facility",  lookup_facility_df, "Course and Facility Look Ups.xlsx"),
]

write_summary = []

for table_name, sdf, source_file in bronze_writes:
    target = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    enriched = (
        sdf
        .withColumn("_source_file", F.lit(source_file).cast(StringType()))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_run_id",      F.lit(run_id))
    )

    row_count = enriched.count()

    # Drop any existing table so the column mapping property
    # is applied fresh at creation. Bronze is rebuildable from
    # raw, so dropping is safe.
    spark.sql(f"DROP TABLE IF EXISTS {target}")

    enriched.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("delta.columnMapping.mode", "name") \
        .saveAsTable(target)

    write_summary.append((table_name, row_count))

    if DEBUG:
        print(f"  Wrote {target}: {row_count} rows")

  Wrote ktu_assessment_dev.bronze.capturing_tool: 11607 rows
  Wrote ktu_assessment_dev.bronze.chw_attendance: 359 rows
  Wrote ktu_assessment_dev.bronze.online_export: 25824 rows
  Wrote ktu_assessment_dev.bronze.lookup_courses: 340 rows
  Wrote ktu_assessment_dev.bronze.lookup_facility: 809 rows


In [0]:
# ============================================================
# BRONZE VERIFICATION
# ============================================================
#
# WHAT THIS CELL DOES:
# Confirms every Bronze table exists, reports its row count,
# and verifies the lineage columns are populated.
#
# THIS IS THE EVIDENCE A REVIEWER USES:
#   - Bronze tables reflect the source row counts
#   - Lineage columns are present and populated
#   - No rows were lost or duplicated during ingestion
#
# WHY DESCRIBE AND NOT SELECT *:
# The capturing tool has 37 columns and CHW has 49.
# Printing full tables would flood the output. DESCRIBE
# shows the schema only. Row counts are read separately.
# ============================================================

if DEBUG:
    print("=" * 70)
    print("BRONZE VERIFICATION")
    print("=" * 70)
    print()

    for table_name, _, _ in bronze_writes:
        full_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
        count = spark.sql(f"SELECT COUNT(*) AS n FROM {full_name}").collect()[0]["n"]
        null_run_id = spark.sql(f"""
            SELECT COUNT(*) AS n FROM {full_name} WHERE _run_id IS NULL
        """).collect()[0]["n"]

        print(f"  {full_name:<50} {count:>7} rows   null_run_id={null_run_id}")

    print()
    print("=" * 70)
    print("BRONZE SCHEMA SAMPLE: capturing_tool")
    print("=" * 70)
    spark.sql(
        f"DESCRIBE {CATALOG}.{BRONZE_SCHEMA}.capturing_tool"
    ).show(50, truncate=False)

    print("=" * 70)
    print("BRONZE SCHEMA SAMPLE: chw_attendance")
    print("=" * 70)
    spark.sql(
        f"DESCRIBE {CATALOG}.{BRONZE_SCHEMA}.chw_attendance"
    ).show(60, truncate=False)

BRONZE VERIFICATION

  ktu_assessment_dev.bronze.capturing_tool             11607 rows   null_run_id=0
  ktu_assessment_dev.bronze.chw_attendance               359 rows   null_run_id=0
  ktu_assessment_dev.bronze.online_export              25824 rows   null_run_id=0
  ktu_assessment_dev.bronze.lookup_courses               340 rows   null_run_id=0
  ktu_assessment_dev.bronze.lookup_facility              809 rows   null_run_id=0

BRONZE SCHEMA SAMPLE: capturing_tool
+---------------------------------------------------------------------------------+---------+-------+
|col_name                                                                         |data_type|comment|
+---------------------------------------------------------------------------------+---------+-------+
|Captured By                                                                      |string   |NULL   |
|Date first captured (YYYY/MM/DD)                                                 |string   |NULL   |
|Course Name         

In [0]:
# ============================================================
# FINALISE AUDIT
# ============================================================
#
# WHAT THIS CELL DOES:
# Updates the RUNNING row created in Cell 2 with the final
# outcome, row counts, and duration.
#
# WHY UPDATE AND NOT INSERT:
# Cell 2 inserted a RUNNING row so the audit trail shows
# the notebook started. This cell closes that same row
# with the final outcome, so the run appears as one
# complete record rather than two disconnected entries.
# ============================================================

end_time = datetime.now()
duration = int((end_time - start_time).total_seconds())

total_rows = sum(count for _, count in write_summary)

message = (
    f"Bronze layer complete. {len(write_summary)} tables written, "
    f"{total_rows} total rows."
)

spark.sql(f"""
    UPDATE {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    SET
        end_time         = '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
        status           = 'SUCCESS',
        rows_in          = {total_rows},
        rows_out         = {total_rows},
        rows_rejected    = 0,
        message          = '{message}',
        duration_seconds = {duration}
    WHERE run_id = '{run_id}'
""")

if DEBUG:
    print()
    print("=" * 50)
    print("BRONZE SUMMARY")
    print("=" * 50)
    print(f"Run ID             : {run_id}")
    print(f"Tables written     : {len(write_summary)}")
    print(f"Total rows         : {total_rows}")
    print(f"Duration           : {duration}s")
    print(f"Status             : SUCCESS")
    print("=" * 50)

print("nb_02_Bronze completed successfully.")


BRONZE SUMMARY
Run ID             : a74a64a6-7c9f-45a8-ad11-441503991a26
Tables written     : 5
Total rows         : 38939
Duration           : 476s
Status             : SUCCESS
nb_02_Bronze completed successfully.


In [0]:
# ============================================================
# SPOT-CHECK BRONZE CONTENT
# ============================================================
#
# WHAT THIS CELL DOES:
# Displays a small sample of rows from each fact source to
# confirm the ingestion read the correct sheet with the
# correct header row.
#
# WHY THIS CHECK MATTERS:
# A wrong header_row silently produces columns named
# 'Unnamed: 1' or the first data row as column names. The
# Jupyter profiling caught this for the CHW files. This
# check confirms the Databricks ingestion read the same
# way.
# ============================================================

if DEBUG:
    print("=" * 70)
    print("SPOT-CHECK: CAPTURING TOOL (first 3 rows, key columns)")
    print("=" * 70)
    spark.sql(f"""
        SELECT
            `Course Name`,
            `Participant Surname`,
            `Participant First name`,
            `Profession`,
            `Facility`,
            `District`,
            `Attendance status`,
            _source_sheet
        FROM {CATALOG}.{BRONZE_SCHEMA}.capturing_tool
        LIMIT 3
    """).show(truncate=False)

    print("=" * 70)
    print("SPOT-CHECK: CHW ATTENDANCE (first 3 rows, key columns)")
    print("=" * 70)
    spark.sql(f"""
        SELECT
            `Participant Surname`,
            `Participant First name`,
            `Profession`,
            `Facility`,
            `District`,
            _source_sheet
        FROM {CATALOG}.{BRONZE_SCHEMA}.chw_attendance
        LIMIT 3
    """).show(truncate=False)

    print("=" * 70)
    print("SPOT-CHECK: ONLINE EXPORT (first 3 rows, key columns)")
    print("=" * 70)
    spark.sql(f"""
        SELECT
            `Course Name`,
            `Last Name`,
            `First Name`,
            `Profession`,
            `Location Name`,
            `District`,
            _source_sheet
        FROM {CATALOG}.{BRONZE_SCHEMA}.online_export
        LIMIT 3
    """).show(truncate=False)

SPOT-CHECK: CAPTURING TOOL (first 3 rows, key columns)
+----------------+-------------------+----------------------+-------------+--------------------------------------+------------+-----------------+-------------+
|Course Name     |Participant Surname|Participant First name|Profession   |Facility                              |District    |Attendance status|_source_sheet|
+----------------+-------------------+----------------------+-------------+--------------------------------------+------------+-----------------+-------------+
|Foundational CPR|NameAacfbcaedf     |NameFbebeececf        |Student nurse|Western Cape College of Nursing (WCCN)|Garden Route|Attended         |Capturer2    |
|Foundational CPR|NameBdedadccae     |NameDabfaccbad        |Student nurse|Western Cape College of Nursing (WCCN)|Garden Route|Attended         |Capturer2    |
|Foundational CPR|NameBfccddbdbd     |NameDebeceebbc        |Student nurse|Western Cape College of Nursing (WCCN)|Garden Route|Attended         |